In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 
import plotly.graph_objs as go
import plotly.figure_factory as ff
from plotly import tools
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot

# Loading the Data

In [ ]:
df=pd.read_csv("/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv")

# Look at data

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
df.describe()

# Checking for missing data

In [ ]:
missing_data=df.isnull().sum().sort_values(ascending=False)
missing_data

# No missing values

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


sns.countplot(x="Class",hue="Class", data=df, palette='viridis')

plt.title('Distribution of Binary Column')
plt.xlabel('Value (0 or 1)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
percentage = df['Class'].value_counts(normalize=True) * 100
print(percentage)

# Highly imbalanced 

# Data Exploration

## Transactions at time

In [ ]:
class_0 = df.loc[df['Class'] == 0]["Time"]
class_1 = df.loc[df['Class'] == 1]["Time"]

hist_data = [class_0, class_1]
group_labels = ['Not Fraud', 'Fraud']

fig = ff.create_distplot(hist_data, group_labels, show_hist=True, show_rug=False)
fig['layout'].update(title='Credit Card Transactions Time Density Plot', xaxis=dict(title='Time [s]'))
iplot(fig, filename='dist_only')

# Transaction Amount

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12,6))
s = sns.boxplot(ax = ax1, x="Class", y="Amount", hue="Class",data=df, palette="PRGn",showfliers=True)
s = sns.boxplot(ax = ax2, x="Class", y="Amount", hue="Class",data=df, palette="PRGn",showfliers=False)
plt.show();

In [ ]:
tmp = df[['Amount','Class']].copy()
class_0 = tmp.loc[tmp['Class'] == 0]['Amount']
class_1 = tmp.loc[tmp['Class'] == 1]['Amount']
class_0.describe()

In [ ]:
class_1.describe()

# Now fraud transaction against time

In [ ]:
fraud = df.loc[df['Class'] == 1]

trace = go.Scatter(
    x = fraud['Time'],y = fraud['Amount'],
    name="Amount",
     marker=dict(
                color='rgb(238,23,11)',
                line=dict(
                    color='red',
                    width=1),
                opacity=0.5,
            ),
    text= fraud['Amount'],
    mode = "markers"
)
data = [trace]
layout = dict(title = 'Amount of fraudulent transactions',
          xaxis = dict(title = 'Time [s]', showticklabels=True), 
          yaxis = dict(title = 'Amount'),
          hovermode='closest'
         )
fig = dict(data=data, layout=layout)
iplot(fig, filename='fraud-amount')

In [ ]:
plt.figure(figsize = (14,14))
plt.title('Credit Card Transactions features correlation plot (Pearson)')
corr = df.corr()
sns.heatmap(corr,xticklabels=corr.columns,yticklabels=corr.columns,linewidths=.1,cmap="Reds")
plt.show()

In [ ]:
target = 'Class'
predictors = ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',\
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19',\
       'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28',\
       'Amount']

In [ ]:
import gc
from datetime import datetime 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score




In [ ]:
df.head()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
# 1. IMPORTS
# ==============================
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# 4. PREPROCESSING  --> first spliting then scalling
# ==============================
scaler = StandardScaler()

df = df.drop(['Time'], axis=1)

X = df.drop('Class', axis=1)
y = df['Class']



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# THEN fit scaler on train only, transform both
# fit → Train data nu mean ane std deviation yaad kare (e.g. mean=88, std=250)
# transform → Aa formula lagave: (value - mean) / std
scaler = StandardScaler()
X_train['normAmount'] = scaler.fit_transform(X_train['Amount'].values.reshape(-1, 1))

# fit nathi kartu — train vali j mean/std vaaparase test par pan. Aa important che:
#Real world ma navu data aave tyare tmare paase 
#train set ni j statistics hoy — test set ni nahi.
#Isliye test par sirf transform j karo.
X_test['normAmount']  = scaler.transform(X_test['Amount'].values.reshape(-1, 1))

X_train = X_train.drop('Amount', axis=1)
X_test  = X_test.drop('Amount', axis=1)

In [ ]:
# ==============================
# 5. CLASS WEIGHTS (FIXED)
# ==============================
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight = dict(enumerate(class_weights))# Result:
# {0: 0.5, 1: 289.0}

print("Class Weights:", class_weight)



Credit card fraud dataset ma typically:

Not Fraud (0) → 284,000 transactions  (99.8%)
Fraud (1)     →     492 transactions  (0.2%)

Model ne train karso toh oo sirf 0 predict karse ane 99.8% accurate laagse — pan koi kaam nahi

Step 1 — Weights calculate karo
pythonclass_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
Andar thi aa formula chale che:
weight = total_samples / (n_classes × samples_in_class)

Fraud weight    = 284492 / (2 × 492)   = ~289
Non-fraud weight= 284492 / (2 × 284000) = ~0.5
Matlab — fraud sample ni ek mistake = 289 normal mistakes jitni costly model ne laagse.

In [ ]:

# ==============================
# 6. MODEL
# ==============================
def create_model():
    initializer = tf.keras.initializers.HeNormal()

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train.shape[1],)),

        tf.keras.layers.Dense(256, activation='relu', kernel_initializer=initializer),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(256, activation='relu', kernel_initializer=initializer),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(256, activation='relu', kernel_initializer=initializer),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),

        # ✅ Better for imbalanced — class_weight sathe pan vadhu effective --> Focal Loss
   
       loss = tf.keras.losses.BinaryFocalCrossentropy(
            apply_class_balancing=False, # Set to True to use the alpha parameter                 # Weight for the minority class (fraud)
            gamma=2.0,                 # Focusing parameter for hard examples
            from_logits=False 
           # aplha=0.998 remove kar diya kyu ki apha ki value se minority claass ka weightage badhaya ja sakta tha par hamne vo class weight me alag se kar liya hai vo bhi strongly
           #alpha = fraud_weight / (fraud_weight + normal_weight)
#alpha = 289.0 / (289.0 + 0.5)
#alpha = 289.0 / 289.5
#alpha ≈ 0.998
       ),

        metrics=[
            tf.keras.metrics.AUC(name='roc_auc'),
            tf.keras.metrics.AUC(curve='PR', name='pr_auc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision')
        ]
    )

    return model


In [ ]:
# 7. CROSS VALIDATION
# ==============================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_auc = []



cv_thresholds = []
cv_auc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

    print(f"\n🔥 Training Fold {fold}")

    X_f_train = X_train.iloc[train_idx]
    y_f_train = y_train.iloc[train_idx]

    X_f_val = X_train.iloc[val_idx]
    y_f_val = y_train.iloc[val_idx]

    model = create_model()

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_pr_auc',  # because imbalanced me auc-roc is worthless
        mode='max',
        patience=5,
        restore_best_weights=True
    )
    
    model.fit(
        X_f_train, y_f_train,
        validation_data=(X_f_val, y_f_val),
        epochs=50,
        batch_size=256,
        class_weight=class_weight,
        callbacks=[early_stop],
        verbose=1
    )

    # 🔥 Predict on validation set
    y_val_prob = model.predict(X_f_val)

    # 🔥 Find best threshold (NO TEST DATA USED)
    precision_cv, recall_cv, thresholds_cv = precision_recall_curve(y_f_val, y_val_prob)

    f1_scores = 2 * (precision_cv * recall_cv) / (precision_cv + recall_cv + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds_cv[best_idx]

    cv_thresholds.append(best_threshold)

    # AUC
    val_auc = roc_auc_score(y_f_val, y_val_prob)
    cv_auc.append(val_auc)

    print(f"Fold {fold} AUC: {val_auc:.4f}")
    print(f"Fold {fold} Best Threshold: {best_threshold:.4f}")

In [ ]:
final_threshold = np.mean(cv_thresholds)

print("\n Final Threshold from CV:", final_threshold)
print("Average CV AUC:", np.mean(cv_auc))

In [ ]:
# 8. FINAL TRAIN ON FULL TRAIN DATA ,before during cross validation it was trained on 80 percent data as kford=5
# ==============================

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train    # shuffle automatic + fraud ratio maintain
)

early_stop_final = tf.keras.callbacks.EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=5,
    restore_best_weights=True
)

model = create_model()

model.fit(
    X_train, y_train,
    validation_data=(X_val,y_val),
    epochs=50,
    batch_size=256,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
# 10. FINAL PREDICTIONS
# ==============================
y_pred_prob=model.predict(X_test)

# Use CV threshold (NOT test tuned)

y_pred = (y_pred_prob > final_threshold).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ==============================

In [ ]:


# 11. CONFUSION MATRIX
# ==============================
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Fraud'],
            yticklabels=['Normal','Fraud'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ==============================
# 12. PR CURVE
# ==============================
precision_test, recall_test, _ = precision_recall_curve(y_test, y_pred_prob)
avg_precision = average_precision_score(y_test, y_pred_prob)

plt.figure(figsize=(8,6))
plt.plot(recall_test, precision_test, label=f'PR AUC = {avg_precision:.4f}')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid()
plt.show()


keras_model     = model           # ← rename
keras_threshold = final_threshold  # ← rename

# Now we will train XGBoost for roughly same pipeline

In [ ]:
# ==============================
# 6. MODEL — XGBoost
# ==============================
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, precision_recall_curve,
                             average_precision_score, classification_report,
                             confusion_matrix)

# Class imbalance handle karvaa
# compute_class_weight ni jagah scale_pos_weight use thay XGBoost ma
fraud_count     = (y_train == 1).sum()
non_fraud_count = (y_train == 0).sum()
scale = non_fraud_count / fraud_count
print(f"scale_pos_weight: {scale:.2f}")  # ~289

def create_xgb_model():
    return XGBClassifier(
        n_estimators=500,
        learning_rate=0.01,
        max_depth=6,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale,   # ← class_weight ni jagah
        eval_metric='aucpr',      # ← PR-AUC monitor karse
        early_stopping_rounds=20, # ← EarlyStopping built-in che
        random_state=42,
        tree_method='hist'        # ← fast training
    )

# ==============================
# 7. CROSS VALIDATION
# ==============================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_thresholds = []
cv_auc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    print(f"\nTraining Fold {fold}...")

    X_f_train = X_train.iloc[train_idx]
    y_f_train = y_train.iloc[train_idx]
    X_f_val   = X_train.iloc[val_idx]
    y_f_val   = y_train.iloc[val_idx]

    model = create_xgb_model()

    model.fit(
        X_f_train, y_f_train,
        eval_set=[(X_f_val, y_f_val)],  # ← validation_data ni jagah
        verbose=True
    )

    y_val_prob = model.predict_proba(X_f_val)[:, 1]  # ← .predict() nahi, proba

    # Best threshold — F1 maximize
    precision_cv, recall_cv, thresholds_cv = precision_recall_curve(
        y_f_val, y_val_prob
    )
    f1_scores = 2 * (precision_cv * recall_cv) / (precision_cv + recall_cv + 1e-8)
    best_idx       = np.argmax(f1_scores)
    best_threshold = thresholds_cv[best_idx]
    cv_thresholds.append(best_threshold)

    val_auc = roc_auc_score(y_f_val, y_val_prob)
    cv_auc.append(val_auc)

    print(f"Fold {fold} AUC:       {val_auc:.4f}")
    print(f"Fold {fold} Threshold: {best_threshold:.4f}")

final_threshold = np.mean(cv_thresholds)
print("\nFinal Threshold from CV:", final_threshold)
print("Average CV AUC:         ", np.mean(cv_auc))

# ==============================
# 8. FINAL TRAINING
# ==============================
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

model = create_xgb_model()
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100   # har 100 trees par print karse
)

# ==============================
# 10. FINAL PREDICTIONS
# ==============================
y_pred_prob = model.predict_proba(X_test)[:, 1]
y_pred      = (y_pred_prob > final_threshold).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ==============================
# 11. CONFUSION MATRIX
# ==============================
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Fraud'],
            yticklabels=['Normal','Fraud'])
plt.title("Confusion Matrix — XGBoost")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ==============================
# 12. PR CURVE
# ==============================
precision_test, recall_test, _ = precision_recall_curve(y_test, y_pred_prob)
avg_precision = average_precision_score(y_test, y_pred_prob)

plt.figure(figsize=(8,6))
plt.plot(recall_test, precision_test,
         label=f'PR AUC = {avg_precision:.4f}')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — XGBoost")
plt.legend()
plt.grid()
plt.show()

# ==============================
# 13. FEATURE IMPORTANCE
# ==============================
import pandas as pd

feat_imp = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)[:15]

plt.figure(figsize=(8,6))
feat_imp.plot(kind='bar')
plt.title("Top 15 Feature Importance — XGBoost")
plt.ylabel("Importance Score")
plt.tight_layout()
plt.show()

xgb_model     = model           # ← rename
xgb_threshold = final_threshold  # ← rename

# Comparison 

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix)

# ==============================
# KERAS PREDICTIONS
# ==============================
keras_prob = model.predict(X_test)
keras_pred = (keras_prob > final_threshold).astype(int)

# ==============================
# XGBOOST PREDICTIONS  
# ==============================
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_pred = (xgb_prob > xgb_threshold).astype(int)

# ==============================
# COMPARISON
# ==============================
models = {
    'Keras Neural Net': (keras_prob, keras_pred),
    'XGBoost':          (xgb_prob,   xgb_pred)
}

print("=" * 60)
for name, (prob, pred) in models.items():
    print(f"\n{name}")
    print("-" * 40)
    
    # 1. ROC-AUC
    roc = roc_auc_score(y_test, prob)
    
    # 2. PR-AUC — MOST IMPORTANT for fraud
    pr = average_precision_score(y_test, prob)
    
    # 3. Classification report
    report = classification_report(y_test, pred, 
                                   target_names=['Normal', 'Fraud'],
                                   output_dict=True)
    
    fraud = report['Fraud']
    
    print(f"ROC-AUC:        {roc:.4f}")
    print(f"PR-AUC:         {pr:.4f}  ← MAIN METRIC")
    print(f"Fraud Precision: {fraud['precision']:.4f}")
    print(f"Fraud Recall:    {fraud['recall']:.4f}")
    print(f"Fraud F1:        {fraud['f1-score']:.4f}")

print("=" * 60)

# Here Roc-AUC is misleading for imbalanced data so we use PR-AUC(Search kari jo)
# 
# PR-AUC of NN waq very less(pehle mai apha=0.25 aur class weight dono use kar raha tha)  as recall to saro che pan precison kharab che --> maine thiki kar diya
# Fraud vagere ma Minimum Precision = 30-40%  →  review team handle kari shake
#                 Maximum Recall    = 85-90%  →  major fraud pakdaay

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ==============================
# 1. PR CURVE — DONO EK SAATH
# ==============================
ax = axes[0]
for name, (prob, pred) in models.items():
    p, r, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    ax.plot(r, p, label=f'{name} (AP={ap:.3f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('PR Curve Comparison')
ax.legend()
ax.grid()

# ==============================
# 2. CONFUSION MATRIX — DONO
# ==============================
for i, (name, (prob, pred)) in enumerate(models.items()):
    ax = axes[i+1]
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Fraud'],
                yticklabels=['Normal', 'Fraud'])
    ax.set_title(f'Confusion Matrix\n{name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

# Agar banne na best leva hoy

In [ ]:
# Dono na probabilities average karo
ensemble_prob = (keras_prob + xgb_prob.reshape(-1, 1)) / 2
ensemble_prob_flat = ensemble_prob.flatten()

# Threshold
precision_e, recall_e, thresholds_e = precision_recall_curve(
    y_test, ensemble_prob_flat
)
f1_scores = 2 * (precision_e * recall_e) / (precision_e + recall_e + 1e-8)
best_idx = np.argmax(f1_scores)
ensemble_threshold = thresholds_e[best_idx]
print(f"Ensemble Threshold: {ensemble_threshold:.4f}")

ensemble_pred = (ensemble_prob_flat > ensemble_threshold).astype(int)

# ✅ PR-AUC
pr_auc_ensemble = average_precision_score(y_test, ensemble_prob_flat)
print(f"Ensemble PR-AUC:  {pr_auc_ensemble:.4f}")

# ✅ ROC-AUC
roc_auc_ensemble = roc_auc_score(y_test, ensemble_prob_flat)
print(f"Ensemble ROC-AUC: {roc_auc_ensemble:.4f}")

# ✅ Fraud specific metrics
report = classification_report(y_test, ensemble_pred,
                                target_names=['Normal', 'Fraud'],
                                output_dict=True,
                                zero_division=0)

fraud = report['Fraud']
print(f"Fraud Precision:  {fraud['precision']:.4f}")
print(f"Fraud Recall:     {fraud['recall']:.4f}")
print(f"Fraud F1:         {fraud['f1-score']:.4f}")

# ✅ Full report pan
print("\nFull Report:")
print(classification_report(y_test, ensemble_pred,
                             target_names=['Normal', 'Fraud'],
                             zero_division=0))